# Hands-on 27 (Bab 11): Ekspor Model Sederhana ke Format Ringan

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Dua jalur deployment edge yang realistis:
- **Jalur A:** Random Forest scikit-learn -> joblib (untuk Raspberry Pi dengan Python).
- **Jalur B:** MLP kecil Keras -> .keras -> TensorFlow Lite (untuk inferensi edge).

## Tujuan

Fokus praktik ini:

- melatih model sederhana di laptop Ubuntu,
- mengekspor model ke format yang lebih ringan atau lebih mudah dideploy,
- dan menyiapkan model agar siap dipakai pada inferensi lokal.


In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


## 1. Jalur A: Ekspor scikit-learn (joblib)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/export_sklearn_model.py"], capture_output=True, text=True)
print(r.stdout)

Model scikit-learn berhasil disimpan ke: models/room_status_rf.joblib



## 2. Jalur B: Keras -> TFLite

In [3]:
r = subprocess.run([sys.executable, "scripts/export_keras_to_tflite.py"], capture_output=True, text=True)
print(r.stdout)

Epoch 1/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 22s 481ms/step - accuracy: 0.3750 - loss: 9.9634
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 730us/step - accuracy: 0.9199 - loss: 1.0015 
Epoch 2/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.9688 - loss: 0.9523
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - accuracy: 0.9775 - loss: 0.2311
Epoch 3/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9688 - loss: 0.0332
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - accuracy: 0.9801 - loss: 0.2068
Epoch 4/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9688 - loss: 0.1355
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step - accuracy: 0.9795 - loss: 0.2068
Epoch 5/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accur

## 3. Verifikasi Inferensi dari File Model

In [4]:
import joblib
import numpy as np
import pandas as pd

# Inferensi ulang dari model joblib
model = joblib.load("models/room_status_rf.joblib")
X_test = pd.read_csv("data/processed/classification_status/X_test.csv")
print("[INFO] Prediksi joblib (5 baris):", list(model.predict(X_test.head())))

# Inferensi ulang dari model TFLite (batch dim default 1: resize dulu ke 5 baris)
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="models/room_binary.tflite")
Xb = pd.read_csv("data/processed/classification_room/X_test.csv").values.astype("float32")[:5]
inp_detail = interpreter.get_input_details()[0]
interpreter.resize_tensor_input(inp_detail["index"], Xb.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(inp_detail["index"], Xb)
interpreter.invoke()
out = interpreter.get_output_details()[0]["index"]
print("[INFO] Probabilitas TFLite (5 baris):", interpreter.get_tensor(out).flatten().round(3).tolist())

[INFO] Prediksi joblib (5 baris): ['normal', 'normal', 'normal', 'normal', 'normal']


I0000 00:00:1793596494.809980    2532 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793596494.810369    2532 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793596494.838045    2532 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793596495.751815    2532 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793596495.752119    2532 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


[INFO] Probabilitas TFLite (5 baris): [0.9990000128746033, 0.0, 0.0, 0.0, 0.0]


/home/sandbox/.local/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Penutup

Training dan deployment adalah dua tahap berbeda. Model joblib (sekitar 425 KB) siap
untuk Raspberry Pi dengan Python biasa; model TFLite hanya sekitar 3.9 KB dan bisa
berjalan dengan interpreter ringan tanpa TensorFlow penuh. Keduanya terverifikasi bisa
dimuat ulang dan dipakai inferensi dari file hasil ekspor di `models/`.